### Pull the data from the internet

In [ ]:
import pandas as pd
import requests

# Fetch the gender gap data
 "https://ourworldindata.org/grapher/gender-gap-education-levels.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

# Fetch the gender gap metadata
gender_gap_metadata = requests.get("https://ourworldindata.org/grapher/gender-gap-education-levels.metadata.json?v=1&csvType=full&useColumnShortNames=true").json()

# Fetch the UN M49 table data
tables = pd.read_html("https://unstats.un.org/unsd/methodology/m49/overview/")
m49_df = tables[0].copy()  # the main table is typically the first one

# Make column names easier to use
m49_df.columns = [c.strip().lower().replace(" ", "_") for c in m49_df.columns]

# Keep the fields you'll usually want
keep = [
    "country_or_area",
    "m49_code",
    "iso-alpha3_code",
    "iso-alpha2_code",
    "region_name",
    "sub-region_name",
    "intermediate_region_name",
]

m49_df = m49_df[keep].rename(columns={
    "m49_code": "m49",
    "iso-alpha3_code": "iso3",
    "iso-alpha2_code": "iso2",
    "region_name": "region",
    "sub-region_name": "subregion",
    "intermediate_region_name": "intermediate_region",
})

# Clean up
m49_df["iso3"] = m49_df["iso3"].astype(str).str.strip()
m49_df["iso2"] = m49_df["iso2"].astype(str).str.strip()
m49_df["m49"]  = pd.to_numeric(m49_df["m49"], errors="coerce")

# Optional: drop rows without ISO3 (some areas/entries may not have one)
m49_lookup = m49_df.dropna(subset=["iso3"]).drop_duplicates(subset=["iso3"])


In [2]:
# Delete the unneccessary variables
del keep, m49_df, tables 

### Data Prep

In [3]:
# Merge the region and subregion columns from m49 into main df
merge_df = gender_gap_df.merge(
    m49_lookup[["iso3","region","subregion","intermediate_region","m49"]],
    left_on="code",
    right_on="iso3",
    how="left",
    validate="m:1"
)

In [4]:
# Check how many rows don't have a 'code' which matches a 'iso3'
missing_iso3 = merge_df["iso3"].isna().sum()
print(f"Number of rows without a matching ISO3 code: {missing_iso3}")

# Print countries missing region/subregion info
missing_region = merge_df[merge_df["region"].isna()]["entity"].unique()
print("Countries without region/subregion info:")
for country in missing_region:
    print(f"- {country}")  


Number of rows without a matching ISO3 code: 1011
Countries without region/subregion info:
- Africa
- Asia
- Central and Southern Asia (SDG)
- East Asia and Pacific (WB)
- Eastern and South-Eastern Asia (SDG)
- Europe
- Europe and Central Asia (WB)
- Europe and Northern America (SDG)
- European Union (27)
- High-income countries
- Latin America and Caribbean (WB)
- Latin America and the Caribbean (SDG)
- Low-income countries
- Lower-middle-income countries
- Middle East and North Africa (WB)
- Middle-income countries
- North America
- North America (WB)
- Northern Africa and Western Asia (SDG)
- Oceania (excluding Australia and New Zealand) (SDG)
- South America
- South Asia (WB)
- Sub-Saharan Africa (SDG)
- Sub-Saharan Africa (WB)
- Taiwan
- Upper-middle-income countries
- World


In [5]:
unique_entities = merge_df["entity"].unique()
for entity in unique_entities:
    print(entity)

Afghanistan
Africa
Albania
Algeria
American Samoa
Andorra
Angola
Anguilla
Antigua and Barbuda
Argentina
Armenia
Aruba
Asia
Australia
Austria
Azerbaijan
Bahamas
Bahrain
Bangladesh
Barbados
Belarus
Belgium
Belize
Benin
Bermuda
Bhutan
Bolivia
Bosnia and Herzegovina
Botswana
Brazil
British Virgin Islands
Brunei
Bulgaria
Burkina Faso
Burundi
Cambodia
Cameroon
Canada
Cape Verde
Cayman Islands
Central African Republic
Central and Southern Asia (SDG)
Chad
Chile
China
Colombia
Comoros
Congo
Cook Islands
Costa Rica
Cote d'Ivoire
Croatia
Cuba
Curacao
Cyprus
Czechia
Democratic Republic of Congo
Denmark
Djibouti
Dominica
Dominican Republic
East Asia and Pacific (WB)
East Timor
Eastern and South-Eastern Asia (SDG)
Ecuador
Egypt
El Salvador
Equatorial Guinea
Eritrea
Estonia
Eswatini
Ethiopia
Europe
Europe and Central Asia (WB)
Europe and Northern America (SDG)
European Union (27)
Fiji
Finland
France
French Guiana
French Polynesia
Gabon
Gambia
Georgia
Germany
Ghana
Gibraltar
Greece
Grenada
Guatemala
G

In [6]:
# Export the original data to a CSV file
gender_gap_df.to_csv("data/gender_gap_education_levels.csv", index=False)
